# FloodSight — Data Collection Notebook
### Douala flood dataset · NASA Space Apps 2026 · "Dancing with the SARs" · Lions Roar Labs

This notebook gathers **every free data source we need** to detect, check and explain floods in Douala, and stores them as one clean, documented dataset in Google Drive.

| # | Source | What it gives us | Role in FloodSight |
|---|---|---|---|
| A | **NISAR L2 GCOV** (ASF DAAC) | Radar backscatter HH + HV, every 12 days | **Core data** — flood detection |
| B | **OPERA DSWx-S1** (PO.DAAC) | Ready-made water maps from Sentinel-1, 30 m | Independent **validation** of our flood maps |
| C | **OPERA RTC-S1** (ASF DAAC) | Sentinel-1 backscatter (VV/VH) | Optional **gap-filler** between NISAR dates |
| D | **GPM IMERG Late daily** (GES DISC) | Daily rainfall (mm/day), 0.1° | Tells us **which dates should show floods** |
| E | **Copernicus DEM 30 m** (Planetary Computer) | Elevation | Low ground floods first; removes false alarms on slopes |
| F | **JRC Global Surface Water** (Planetary Computer) | Long-term water occurrence 1984–2021 | Reference for **permanent water** (Wouri, estuary) |
| G | **ESA WorldCover** (Planetary Computer) | Land cover (built-up, trees, wetland…) | Explains signal types (city vs. mangrove vs. farmland) |
| H | **OpenStreetMap** | Neighbourhoods, roads, waterways | **Impact**: which areas were hit |
| I | **Flood reports** (news, ReliefWeb) | Real flood dates + places | Ground truth for the pitch (filled by hand) |

**How to use**
1. 🔑 Secrets panel (left) → add `EARTHDATA_USERNAME`, `EARTHDATA_PASSWORD` → turn on *Notebook access*
2. *Runtime → Run all* (allow Google Drive when asked)
3. Every section is **independent and re-runnable**: finished files are skipped, a failing source never stops the others
4. At the end, `catalog/manifest.json` lists every file with its source, date and citation

## 0 · Setup

In [ ]:
!pip -q install asf_search earthaccess "osmnx>=2.0" pystac-client planetary-computer rioxarray rasterio h5py h5netcdf pyproj geopandas pandas matplotlib

In [ ]:
# ───────────── SETTINGS (edit here only) ─────────────
AOI_NAME = "douala"
BBOX     = (9.60, 3.95, 9.80, 4.15)               # min_lon, min_lat, max_lon, max_lat
AOI_WKT  = "POLYGON((9.60 3.95, 9.80 3.95, 9.80 4.15, 9.60 4.15, 9.60 3.95))"
START, END = "2026-06-01", "2026-10-31"          # flood season window
RAIN_START = "2026-05-15"                        # rainfall starts earlier to see build-up

NISAR_POLS   = ["HHHH", "HVHV"]                  # HH = water, HV = vegetation (bonus)
GET_S1_RTC   = False                             # True = also crop Sentinel-1 backscatter (slower)
USE_DRIVE    = True
# ─────────────────────────────────────────────────────

import os, re, json, time, warnings
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FloodSight/data')
else:
    ROOT = Path('/content/FloodSight/data')

DIRS = {k: ROOT / v for k, v in {
    'nisar': 'nisar/gcov', 'dswx': 'opera_dswx_s1', 's1rtc': 'opera_rtc_s1', 'rain': 'rainfall',
    'static': 'static', 'osm': 'osm', 'reports': 'reports', 'catalog': 'catalog', 'tmp': '_tmp'}.items()}
for p in DIRS.values(): p.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = DIRS['catalog'] / 'manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {'aoi': AOI_NAME, 'bbox': BBOX, 'files': {}}

def record(path, source, citation, **meta):
    """Add/refresh one file in the dataset catalog."""
    rel = str(Path(path).relative_to(ROOT))
    MANIFEST['files'][rel] = {'source': source, 'citation': citation, 'added': datetime.utcnow().isoformat(timespec='seconds') + 'Z', **meta}
    MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2, default=str))

def section(name):
    print('\n' + '═' * 70 + f'\n{name}\n' + '═' * 70)

print('Dataset folder:', ROOT)

In [ ]:
# ───────────── Logins ─────────────
try:
    from google.colab import userdata
    os.environ['EARTHDATA_USERNAME'] = userdata.get('EARTHDATA_USERNAME')
    os.environ['EARTHDATA_PASSWORD'] = userdata.get('EARTHDATA_PASSWORD')
except Exception:
    from getpass import getpass
    os.environ['EARTHDATA_USERNAME'] = input('Earthdata username: ')
    os.environ['EARTHDATA_PASSWORD'] = getpass('Earthdata password: ')

import asf_search as asf, earthaccess
ASF = asf.ASFSession().auth_with_creds(os.environ['EARTHDATA_USERNAME'], os.environ['EARTHDATA_PASSWORD'])
earthaccess.login(strategy='environment')
print('✓ NASA Earthdata login OK (ASF + earthaccess)')

In [ ]:
# ───────────── Shared raster helpers ─────────────
import h5py, rasterio, rioxarray
from rasterio.transform import from_origin
from pyproj import Transformer

def save_tif(arr, transform, crs, path, dtype='float32', nodata=np.nan):
    with rasterio.open(path, 'w', driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype=dtype, crs=crs, transform=transform, nodata=nodata,
                       compress='deflate', tiled=True) as dst:
        dst.write(arr.astype(dtype), 1)

def clip_da(da):
    return da.rio.clip_box(*BBOX, crs='EPSG:4326')

def describe(path):
    with rasterio.open(path) as r:
        return {'crs': str(r.crs), 'shape': [r.height, r.width], 'res_m_or_deg': list(r.res)}

## A · NISAR L2 GCOV — core radar data
Inventory **every** GCOV image over Douala (all tracks), pick the track with the most dates as the **main track**, then stream only the Douala subset of HH and HV for every date and save it in **decibels (dB)**.

In [ ]:
section('A · NISAR GCOV inventory')
CIT_NISAR = 'NISAR L2 GCOV, NASA/JPL & ISRO, distributed by ASF DAAC (https://asf.alaska.edu)'

def parse_nisar_name(name):
    # NISAR_L2_PR_GCOV_<cycle>_<track>_<dir>_<frame>_..._<startTime>_...
    p = name.split('_')
    try:
        return {'config': p[2], 'cycle': int(p[4]), 'track': int(p[5]),
                'direction': {'A': 'ASCENDING', 'D': 'DESCENDING'}.get(p[6], p[6]), 'frame': int(p[7])}
    except Exception:
        return {'config': None, 'cycle': None, 'track': None, 'direction': None, 'frame': None}

res = asf.search(dataset='NISAR', processingLevel='GCOV', intersectsWith=AOI_WKT, start=START, end=END, maxResults=1000)
rows = []
for r in res:
    p = r.properties
    name = p.get('fileID') or p.get('sceneName') or ''
    urls = [p.get('url')] + list(p.get('additionalUrls') or [])
    h5 = [u for u in urls if u and u.endswith('.h5')]
    if not h5: continue
    rows.append({'date': p['startTime'][:10], 'start_time': p['startTime'], 'name': name, 'url': h5[0],
                 'polarization': p.get('polarization'), **parse_nisar_name(name)})
nisar_inv = pd.DataFrame(rows).sort_values(['track', 'frame', 'date']).drop_duplicates(['track', 'frame', 'date', 'config'])
nisar_inv.to_csv(DIRS['catalog'] / 'nisar_gcov_inventory.csv', index=False)
record(DIRS['catalog'] / 'nisar_gcov_inventory.csv', 'NISAR GCOV inventory', CIT_NISAR, n=len(nisar_inv))

summary = nisar_inv.groupby(['track', 'frame', 'direction']).agg(n_dates=('date', 'nunique'), first=('date', 'min'), last=('date', 'max')).reset_index()
MAIN = summary.sort_values('n_dates', ascending=False).iloc[0]
print(f'{len(nisar_inv)} GCOV products over Douala')
print(f"Main track → track {MAIN.track}, frame {MAIN.frame}, {MAIN.direction}: {MAIN.n_dates} dates ({MAIN['first']} → {MAIN['last']})")
summary

In [ ]:
section('A · NISAR GCOV → Douala subset in dB (streamed)')

def crop_gcov(fobj, pols):
    out = {}
    with h5py.File(fobj, 'r') as f:
        for pol in pols:
            found = []
            f.visititems(lambda n, o: found.append(n) if isinstance(o, h5py.Dataset) and n.endswith('/' + pol) and '/grids/' in n else None)
            if not found: continue
            g = f[found[0].rsplit('/', 1)[0]]
            x, y = g['xCoordinates'][()], g['yCoordinates'][()]
            pj = g['projection']; epsg = int(pj.attrs.get('epsg_code', pj[()]))
            minx, miny, maxx, maxy = Transformer.from_crs(4326, epsg, always_xy=True).transform_bounds(*BBOX)
            xi = np.where((x >= minx) & (x <= maxx))[0]; yi = np.where((y >= miny) & (y <= maxy))[0]
            if not len(xi) or not len(yi): continue
            data = g[pol][yi.min():yi.max() + 1, xi.min():xi.max() + 1]
            xs, ys = x[xi.min():xi.max() + 1], y[yi.min():yi.max() + 1]
            dx, dy = float(abs(xs[1] - xs[0])), float(abs(ys[1] - ys[0]))
            if ys[0] < ys[-1]: data, ys = data[::-1], ys[::-1]
            db = 10 * np.log10(np.where(data > 0, data, np.nan))
            out[pol] = (db.astype('float32'), from_origin(xs[0] - dx / 2, ys[0] + dy / 2, dx, dy), f'EPSG:{epsg}')
    return out

todo = nisar_inv  # all tracks; main track first
todo = pd.concat([todo[(todo.track == MAIN.track) & (todo.frame == MAIN.frame)], todo[(todo.track != MAIN.track) | (todo.frame != MAIN.frame)]])

for row in todo.itertuples():
    folder = DIRS['nisar'] / f'track{row.track:03d}_frame{row.frame:03d}_{row.direction[0]}'
    folder.mkdir(exist_ok=True)
    targets = {pol: folder / f'{AOI_NAME}_{row.date}_{pol[:2]}_db.tif' for pol in NISAR_POLS}
    if all(t.exists() for t in targets.values()):
        print('✓ skip', folder.name, row.date); continue
    t0 = time.time()
    try:
        layers = crop_gcov(earthaccess.open([row.url])[0], NISAR_POLS); how = 'streamed'
    except Exception as e:
        print('  stream failed → download:', type(e).__name__)
        fname = row.url.split('/')[-1]
        asf.download_url(row.url, path=str(DIRS['tmp']), filename=fname, session=ASF)
        layers = crop_gcov(str(DIRS['tmp'] / fname), NISAR_POLS); how = 'downloaded'
        (DIRS['tmp'] / fname).unlink(missing_ok=True)
    for pol, (db, tr, crs) in layers.items():
        save_tif(db, tr, crs, targets[pol])
        record(targets[pol], 'NISAR L2 GCOV', CIT_NISAR, date=row.date, track=row.track, frame=row.frame,
               direction=row.direction, polarization=pol, units='dB (gamma0)', product=row.name, **describe(targets[pol]))
    print(f'✓ {folder.name} {row.date} {list(layers)} ({how}, {time.time() - t0:.0f}s)')

## B · OPERA DSWx-S1 — independent water maps (validation)
NASA/JPL's own Sentinel-1 surface-water product (30 m). We compare our NISAR flood maps against it.
Classes in the `WTR` layer: **0** not water · **1/2** open water · **3** inundated vegetation · **250+** masked (see product docs).

In [ ]:
section('B · OPERA DSWx-S1')
CIT_DSWX = 'OPERA Dynamic Surface Water Extent from Sentinel-1 (DSWx-S1) V1, JPL/NASA, PO.DAAC, doi:10.5067/OPDSWS1-L3V1'
try:
    gr = earthaccess.search_data(short_name='OPERA_L3_DSWX-S1_V1', bounding_box=BBOX, temporal=(START, END))
    rows = []
    for g in gr:
        day = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'][:10]
        wtr = [u for u in g.data_links() if u.endswith('_B01_WTR.tif')]
        if wtr: rows.append({'date': day, 'url': wtr[0]})
    dswx_inv = pd.DataFrame(rows).sort_values('date')
    dswx_inv.to_csv(DIRS['catalog'] / 'opera_dswx_s1_inventory.csv', index=False)
    print(f'{len(dswx_inv)} DSWx-S1 water tiles over Douala on {dswx_inv.date.nunique()} dates')

    for day, grp in dswx_inv.groupby('date'):
        out = DIRS['dswx'] / f'{AOI_NAME}_{day}_dswx_wtr.tif'
        if out.exists(): continue
        local = earthaccess.download(list(grp.url), str(DIRS['tmp']))
        parts = []
        for f in local:
            try: parts.append(clip_da(rioxarray.open_rasterio(f).squeeze()))
            except Exception: pass                      # tile does not overlap the AOI
        if not parts: continue
        from rioxarray.merge import merge_arrays
        da = merge_arrays([p.rio.reproject_match(parts[0]) for p in parts]) if len(parts) > 1 else parts[0]
        da.rio.to_raster(out, compress='deflate')
        record(out, 'OPERA DSWx-S1 WTR', CIT_DSWX, date=day, units='class', **describe(out))
        for f in local: Path(f).unlink(missing_ok=True)
        print('✓', out.name)
except Exception as e:
    print('⚠ DSWx-S1 skipped:', e)

## C · OPERA RTC-S1 — Sentinel-1 backscatter (optional gap-filler)

In [ ]:
section('C · OPERA RTC-S1')
CIT_RTC = 'OPERA Radiometric Terrain Corrected SAR Backscatter from Sentinel-1 (RTC-S1), JPL/NASA, ASF DAAC'
try:
    rtc = asf.search(dataset=asf.DATASET.OPERA_S1, processingLevel='RTC', intersectsWith=AOI_WKT, start=START, end=END, maxResults=2000)
    rows = []
    for r in rtc:
        p = r.properties
        urls = [p.get('url')] + list(p.get('additionalUrls') or [])
        vv = [u for u in urls if u and u.endswith('_VV.tif')]
        if vv: rows.append({'date': p['startTime'][:10], 'burst': p.get('fileID'), 'url': vv[0]})
    rtc_inv = pd.DataFrame(rows).sort_values('date')
    rtc_inv.to_csv(DIRS['catalog'] / 'opera_rtc_s1_inventory.csv', index=False)
    print(f'{len(rtc_inv)} RTC-S1 bursts on {rtc_inv.date.nunique()} dates (inventory saved)')
    if GET_S1_RTC:
        for day, grp in rtc_inv.groupby('date'):
            out = DIRS['s1rtc'] / f'{AOI_NAME}_{day}_VV_db.tif'
            if out.exists(): continue
            parts = []
            for u in grp.url:
                fn = u.split('/')[-1]; asf.download_url(u, path=str(DIRS['tmp']), filename=fn, session=ASF)
                try: parts.append(clip_da(rioxarray.open_rasterio(DIRS['tmp'] / fn, masked=True).squeeze()))
                except Exception: pass
                (DIRS['tmp'] / fn).unlink(missing_ok=True)
            if not parts: continue
            da = parts[0] if len(parts) == 1 else parts[0].combine_first(parts[1].rio.reproject_match(parts[0]))
            db = 10 * np.log10(da.where(da > 0))
            db.rio.to_raster(out, compress='deflate')
            record(out, 'OPERA RTC-S1 VV', CIT_RTC, date=day, units='dB (gamma0)', **describe(out))
            print('✓', out.name)
    else:
        print('(Set GET_S1_RTC = True to crop the images too.)')
except Exception as e:
    print('⚠ RTC-S1 skipped:', e)

## D · GPM IMERG — daily rainfall over Douala
Streams only the Douala grid cells from each daily file (no big downloads), then computes:
- daily rainfall (mm/day), and
- **rain in the 3 days before each NISAR pass** → which radar dates should show floods

In [ ]:
section('D · GPM IMERG Late daily rainfall')
import xarray as xr
CIT_IMERG = 'Huffman et al., GPM IMERG Late Precipitation L3 1 day 0.1° V07 (GPM_3IMERGDL), GES DISC, doi:10.5067/GPM/IMERGDL/DAY/07'
rain_csv = DIRS['rain'] / f'{AOI_NAME}_imerg_daily.csv'
try:
    done = pd.read_csv(rain_csv) if rain_csv.exists() else pd.DataFrame(columns=['date', 'rain_mm'])
    gr = earthaccess.search_data(short_name='GPM_3IMERGDL', version='07', temporal=(RAIN_START, END), bounding_box=BBOX)
    gr = [g for g in gr if g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'][:10] not in set(done.date)]
    print(f'{len(gr)} new daily files to read')
    rows = []
    files = earthaccess.open(gr) if gr else []
    for g, f in zip(gr, files):
        day = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'][:10]
        with xr.open_dataset(f, engine='h5netcdf') as ds:
            v = ds['precipitation'].sel(lon=slice(BBOX[0] - 0.05, BBOX[2] + 0.05), lat=slice(BBOX[1] - 0.05, BBOX[3] + 0.05))
            rows.append({'date': day, 'rain_mm': round(float(v.mean()), 2)})
    rain = pd.concat([done, pd.DataFrame(rows)]).drop_duplicates('date').sort_values('date')
    rain.to_csv(rain_csv, index=False)
    record(rain_csv, 'GPM IMERG Late daily (area mean over Douala)', CIT_IMERG, units='mm/day', n_days=len(rain))

    # rain before each NISAR date (main track)
    r = rain.set_index(pd.to_datetime(rain.date)).rain_mm
    mt = nisar_inv[(nisar_inv.track == MAIN.track) & (nisar_inv.frame == MAIN.frame)].drop_duplicates('date')
    link = pd.DataFrame({'nisar_date': mt.date.values})
    link['rain_day_of_mm']   = [r.get(pd.Timestamp(d), np.nan) for d in link.nisar_date]
    link['rain_prev3d_mm']   = [r[pd.Timestamp(d) - pd.Timedelta(days=3):pd.Timestamp(d)].sum() for d in link.nisar_date]
    link['rain_prev7d_mm']   = [r[pd.Timestamp(d) - pd.Timedelta(days=7):pd.Timestamp(d)].sum() for d in link.nisar_date]
    link['expected_wet']     = link.rain_prev3d_mm >= link.rain_prev3d_mm.quantile(0.6)
    link.to_csv(DIRS['rain'] / 'rain_before_nisar_dates.csv', index=False)
    record(DIRS['rain'] / 'rain_before_nisar_dates.csv', 'Rainfall before each NISAR pass', CIT_IMERG)
    display(link)
except Exception as e:
    print('⚠ IMERG skipped:', e)

## E–G · Static layers from Microsoft Planetary Computer (no login needed)
Copernicus DEM (elevation), JRC Global Surface Water (water occurrence 1984–2021), ESA WorldCover (land cover).

In [ ]:
section('E–G · DEM, JRC surface water, land cover')
import pystac_client, planetary_computer
from rioxarray.merge import merge_arrays
cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)

STATIC = [
    ('cop-dem-glo-30', 'data',       'dem_cop30_m.tif',           'Copernicus DEM GLO-30, © DLR/Airbus, ESA Copernicus', 'm'),
    ('jrc-gsw',        'occurrence', 'jrc_gsw_occurrence_pct.tif', 'Pekel et al. 2016, JRC Global Surface Water (EC JRC/Google)', '% of time water'),
    ('esa-worldcover', 'map',        'esa_worldcover_class.tif',  'ESA WorldCover 10 m (Zanaga et al.), © ESA', 'class'),
]
for coll, asset, fname, cit, units in STATIC:
    out = DIRS['static'] / f'{AOI_NAME}_{fname}'
    if out.exists(): print('✓ skip', out.name); continue
    try:
        items = list(cat.search(collections=[coll], bbox=BBOX).items())
        if coll == 'esa-worldcover':
            items = sorted(items, key=lambda i: i.datetime or datetime(1900, 1, 1), reverse=True)[:4]
        parts = [clip_da(rioxarray.open_rasterio(i.assets[asset].href).squeeze()) for i in items if asset in i.assets]
        parts = [p for p in parts if p.size]
        da = merge_arrays(parts) if len(parts) > 1 else parts[0]
        da.rio.to_raster(out, compress='deflate')
        record(out, f'Planetary Computer: {coll}', cit, units=units, **describe(out))
        print('✓', out.name, da.shape)
    except Exception as e:
        print(f'⚠ {coll} skipped:', e)

## H · OpenStreetMap — neighbourhoods, roads, waterways

In [ ]:
section('H · OpenStreetMap')
import osmnx as ox, geopandas as gpd
CIT_OSM = '© OpenStreetMap contributors, ODbL'
bbox_ox = (BBOX[0], BBOX[1], BBOX[2], BBOX[3])          # (left, bottom, right, top) for osmnx ≥ 2
LAYERS = {
    'places':     {'place': ['suburb', 'neighbourhood', 'quarter', 'village', 'city']},
    'admin':      {'boundary': 'administrative'},
    'roads':      {'highway': ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']},
    'waterways':  {'waterway': ['river', 'stream', 'canal', 'drain']},
    'water':      {'natural': 'water'},
}
for name, tags in LAYERS.items():
    out = DIRS['osm'] / f'{AOI_NAME}_osm_{name}.geojson'
    if out.exists(): print('✓ skip', out.name); continue
    try:
        gdf = ox.features_from_bbox(bbox=bbox_ox, tags=tags)
        keep = [c for c in ['name', 'name:fr', 'name:en', 'place', 'admin_level', 'highway', 'waterway', 'natural', 'geometry'] if c in gdf.columns]
        gdf = gdf[keep].reset_index(drop=True)
        gdf.to_file(out, driver='GeoJSON')
        record(out, f'OpenStreetMap: {name}', CIT_OSM, features=len(gdf))
        print(f'✓ {out.name}: {len(gdf)} features')
    except Exception as e:
        print(f'⚠ OSM {name} skipped:', e)

## I · Flood reports (ground truth, filled by hand)
Creates a template. Add one row per real flood event you find in local news or [ReliefWeb](https://reliefweb.int): date, neighbourhood, source link. We will check our radar detections against it.

In [ ]:
section('I · Flood reports template')
rep = DIRS['reports'] / 'flood_reports_douala.csv'
if not rep.exists():
    pd.DataFrame(columns=['date', 'neighbourhood', 'description', 'source_name', 'source_url']).to_csv(rep, index=False)
    record(rep, 'Manual flood reports (team)', 'Compiled by Lions Roar Labs from cited sources')
print('Template:', rep, '→ open it in Google Sheets/Drive and add rows')

## Quality check · what we collected

In [ ]:
section('QA · dataset overview')
import matplotlib.pyplot as plt, matplotlib.dates as mdates

files = pd.DataFrame([{'file': k, **v} for k, v in MANIFEST['files'].items()])
print(files.groupby('source').size().rename('files').to_string())

fig, ax = plt.subplots(figsize=(14, 4.5))
rain_csv = DIRS['rain'] / f'{AOI_NAME}_imerg_daily.csv'
if rain_csv.exists():
    rain = pd.read_csv(rain_csv, parse_dates=['date'])
    ax.bar(rain.date, rain.rain_mm, color='#8FA9C9', width=1, label='Daily rain (mm, IMERG)')
for d in pd.to_datetime(nisar_inv[(nisar_inv.track == MAIN.track) & (nisar_inv.frame == MAIN.frame)].date.unique()):
    ax.axvline(d, color='#2EC4E6', lw=2)
dswx_dates = sorted({p.name.split('_')[1] for p in DIRS['dswx'].glob('*_dswx_wtr.tif')})
for d in pd.to_datetime(dswx_dates):
    ax.axvline(d, color='#F5B700', lw=1, ls='--')
ax.plot([], [], color='#2EC4E6', lw=2, label=f'NISAR pass (track {MAIN.track})')
ax.plot([], [], color='#F5B700', lw=1, ls='--', label='DSWx-S1 water map')
ax.set_title('Douala 2026: rainfall vs. radar observation dates'); ax.set_ylabel('mm/day')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b')); ax.legend(loc='upper left'); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig(DIRS['catalog'] / 'timeline.png', dpi=120); plt.show()

thumbs = [p for p in [*sorted((DIRS['nisar']).glob('*/*_HH_db.tif'))[:1], *sorted(DIRS['static'].glob('*.tif')), *sorted(DIRS['dswx'].glob('*.tif'))[:1]]]
if thumbs:
    fig, axs = plt.subplots(1, len(thumbs), figsize=(5 * len(thumbs), 5))
    axs = np.atleast_1d(axs)
    for a, p in zip(axs, thumbs):
        da = rioxarray.open_rasterio(p, masked=True).squeeze()
        cmap = 'gray' if '_db' in p.name else ('terrain' if 'dem' in p.name else ('Blues' if 'jrc' in p.name else 'tab20'))
        a.imshow(da.values, cmap=cmap); a.set_title(p.name, fontsize=9); a.axis('off')
    plt.tight_layout(); plt.savefig(DIRS['catalog'] / 'layers_preview.png', dpi=110); plt.show()

In [ ]:
section('Dataset README')
lines = ['# FloodSight Douala dataset', f'Generated {datetime.utcnow():%Y-%m-%d %H:%M} UTC · AOI {BBOX}', '',
         f"Main NISAR track: {MAIN.track} / frame {MAIN.frame} / {MAIN.direction} ({MAIN.n_dates} dates)", '',
         '| File | Source | Date | Units |', '|---|---|---|---|']
for k, v in sorted(MANIFEST['files'].items()):
    lines.append(f"| `{k}` | {v['source']} | {v.get('date', '')} | {v.get('units', '')} |")
lines += ['', '## Citations'] + sorted({f"- {v['citation']}" for v in MANIFEST['files'].values()})
(ROOT / 'README.md').write_text('\n'.join(lines))
print('\n'.join(lines[:12]), '\n…\nSaved', ROOT / 'README.md')

## Folder structure produced
```
FloodSight/data/
├── README.md                     ← auto-generated list of files + citations
├── catalog/                      ← manifest.json, inventories (CSV), timeline.png, layers_preview.png
├── nisar/gcov/trackXXX_frameYYY_A/  ← douala_<date>_HH_db.tif, douala_<date>_HV_db.tif
├── opera_dswx_s1/                ← douala_<date>_dswx_wtr.tif (validation water maps)
├── opera_rtc_s1/                 ← douala_<date>_VV_db.tif (only if GET_S1_RTC = True)
├── rainfall/                     ← douala_imerg_daily.csv, rain_before_nisar_dates.csv
├── static/                       ← DEM, JRC water occurrence, ESA WorldCover
├── osm/                          ← places, admin, roads, waterways, water (GeoJSON)
└── reports/                      ← flood_reports_douala.csv (fill by hand)
```
**Next notebook (02):** water detection on the NISAR stack → flood maps, km² per date, hotspots, validation against DSWx-S1 → web files for the React app.

### Acknowledgments
NASA Earthdata · ASF DAAC · NASA–ISRO NISAR mission · NASA JPL OPERA project · PO.DAAC · GES DISC (GPM IMERG) · ESA Copernicus (DEM, WorldCover, Sentinel-1) · EC JRC / Google (Global Surface Water) · Microsoft Planetary Computer · OpenStreetMap contributors